In [2]:
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord, search_around_sky, Angle
import astropy.units as u

def match_emitters_to_clusters(
    clusters: pd.DataFrame,
    stars: pd.DataFrame,
    *,
    cluster_ra_col: str = "ra",        # RA dos aglomerados (graus)
    cluster_dec_col: str = "dec",      # Dec dos aglomerados (graus)
    cluster_amaj_col: str = "amaj",    # semi-eixo maior (ex.: arcmin) — tratado como raio circular
    stars_ra_col: str = "ra",          # RA das estrelas (graus)
    stars_dec_col: str = "dec",        # Dec das estrelas (graus)
    amaj_unit: str = "arcmin",         # unidade do amaj (ex.: "arcmin", "arcsec", "deg")
    keep_cols_clusters: list | None = None,  # colunas extras do df de aglomerados para levar ao resultado
    keep_cols_stars: list | None = None,     # colunas extras do df de estrelas para levar ao resultado
) -> pd.DataFrame:
    """
    Associa estrelas Hα a aglomerados retornando apenas as que estão dentro do amaj de cada aglomerado.

    Parameters
    ----------
    clusters : pandas.DataFrame
        Tabela de aglomerados com colunas de posição e amaj.
    stars : pandas.DataFrame
        Tabela de estrelas emissoras com colunas de posição.
    cluster_ra_col : str, default "ra"
        Nome da coluna de RA dos aglomerados (em graus).
    cluster_dec_col : str, default "dec"
        Nome da coluna de Dec dos aglomerados (em graus).
    cluster_amaj_col : str, default "amaj"
        Nome da coluna do semi-eixo maior do aglomerado.
        **É interpretado como um raio circular** para a seleção.
    stars_ra_col : str, default "ra"
        Nome da coluna de RA das estrelas (em graus).
    stars_dec_col : str, default "dec"
        Nome da coluna de Dec das estrelas (em graus).
    amaj_unit : str, default "arcmin"
        Unidade de `amaj` (por exemplo: "arcmin", "arcsec" ou "deg").
    keep_cols_clusters : list[str] | None
        Colunas adicionais do DataFrame de aglomerados a incluir no resultado.
    keep_cols_stars : list[str] | None
        Colunas adicionais do DataFrame de estrelas a incluir no resultado.

    Returns
    -------
    pandas.DataFrame
        Uma tabela já mesclada contendo:
        - índices originais (`cluster_index`, `star_index`)
        - separação angular (`sep_arcsec`, `sep_arcmin`)
        - colunas dos aglomerados com prefixo `cluster_`
        - colunas das estrelas com prefixo `star_`

    Notes
    -----
    * `amaj` é tratado como **semi-eixo maior** em unidade dada por `amaj_unit` e
      usado como **raio circular equivalente** para o corte (i.e., sep <= amaj).
    * A rotina usa um raio máximo global (= max(amaj)) em `search_around_sky`
      e depois filtra par-a-par por `amaj` específico de cada aglomerado.
    * Linhas com posições/raios inválidos (NaN, não positivos) são ignoradas.
    """
    # Escolha das colunas para carregar/propagar
    if keep_cols_clusters is None:
        keep_cols_clusters = []
    if keep_cols_stars is None:
        keep_cols_stars = []

    # Filtrar entradas válidas nos aglomerados
    cl_req = [cluster_ra_col, cluster_dec_col, cluster_amaj_col] + keep_cols_clusters
    cl_df = clusters[cl_req].copy()
    cl_df = cl_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[cluster_ra_col, cluster_dec_col, cluster_amaj_col])
    # descarta amaj <= 0
    cl_df = cl_df[cl_df[cluster_amaj_col] > 0].copy()

    # Filtrar entradas válidas nas estrelas
    st_req = [stars_ra_col, stars_dec_col] + keep_cols_stars
    st_df = stars[st_req].copy()
    st_df = st_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[stars_ra_col, stars_dec_col])

    if cl_df.empty or st_df.empty:
        return pd.DataFrame(columns=[
            "cluster_index","star_index","sep_arcsec","sep_arcmin"
        ])

    # Coordenadas
    c_clusters = SkyCoord(ra=cl_df[cluster_ra_col].to_numpy() * u.deg,
                          dec=cl_df[cluster_dec_col].to_numpy() * u.deg)
    c_stars = SkyCoord(ra=st_df[stars_ra_col].to_numpy() * u.deg,
                       dec=st_df[stars_dec_col].to_numpy() * u.deg)

    # Raio máximo global para a busca
    unit = u.Unit(amaj_unit)
    max_amaj = np.nanmax(cl_df[cluster_amaj_col].to_numpy())
    seplimit = Angle(max_amaj, unit)

    # Busca inicial (eficiente)
    idx_cl, idx_st, sep2d, _ = search_around_sky(c_clusters, c_stars, seplimit=seplimit)

    if len(idx_cl) == 0:
        return pd.DataFrame(columns=[
            "cluster_index","star_index","sep_arcsec","sep_arcmin"
        ])

    # Limite específico por aglomerado (par-a-par)
    amaj_per_pair = Angle(cl_df.iloc[idx_cl][cluster_amaj_col].to_numpy(), unit)
    keep = sep2d <= amaj_per_pair

    if not np.any(keep):
        return pd.DataFrame(columns=[
            "cluster_index","star_index","sep_arcsec","sep_arcmin"
        ])

    # Construir resultado
    pairs = pd.DataFrame({
        "cluster_index": cl_df.index.values[idx_cl[keep]],
        "star_index": st_df.index.values[idx_st[keep]],
        "sep_arcsec": sep2d[keep].arcsec,
        "sep_arcmin": sep2d[keep].to(u.arcmin).value,
    })

    # Anexar metadados dos aglomerados e estrelas (com prefixos)
    cl_out = clusters.loc[pairs["cluster_index"]][[cluster_ra_col, cluster_dec_col, cluster_amaj_col] + keep_cols_clusters]
    cl_out = cl_out.add_prefix("cluster_").reset_index(drop=True)

    st_out = stars.loc[pairs["star_index"]][[stars_ra_col, stars_dec_col] + keep_cols_stars]
    st_out = st_out.add_prefix("star_").reset_index(drop=True)

    result = pd.concat([pairs.reset_index(drop=True), cl_out, st_out], axis=1)

    # Ordenar por separação (útil)
    result = result.sort_values("sep_arcmin", ascending=True).reset_index(drop=True)
    return result

In [3]:
bica = pd.read_csv('/home/marina/FullCatalogBica_08_20.csv')

In [5]:
be = pd.read_csv('/home/shared/splus_gaia/notebooks/magellanic_clouds/studying_diff_objs/selecting_stars_below_be.csv')

In [8]:
# clusters_df: colunas ["name","ra","dec","amaj"] (amaj em arcmin, por ex.)
# emitters_df: colunas ["source_id","ra","dec", ...]
out = match_emitters_to_clusters(
    bica,
    be,
    cluster_ra_col="ra_deg",
    cluster_dec_col="dec_deg",
    cluster_amaj_col="amaj",
    stars_ra_col="ra",
    stars_dec_col="dec",
    amaj_unit="arcmin",
    keep_cols_clusters=["Names"],         # traga o nome do aglomerado
    keep_cols_stars=["id","excess", "err_mag_psf_u", "err_mag_psf_j0378", "err_mag_psf_j0395",
       "err_mag_psf_j0410", "err_mag_psf_j0430", "err_mag_psf_g",
       "err_mag_psf_j0515", "err_mag_psf_r", "err_mag_psf_j0660",
       "err_mag_psf_i", "err_mag_psf_j0861", "err_mag_psf_z", "mag_psf_u",
       "mag_psf_j0378", "mag_psf_j0395", "mag_psf_j0410", "mag_psf_j0430",
       "mag_psf_g", "mag_psf_j0515", "mag_psf_r", "mag_psf_j0660", "mag_psf_i",
       "mag_psf_j0861", "mag_psf_z"]  # traga id/atributos da estrela
)

# "out" tem uma linha por estrela-associada, com "cluster_*" e "star_*"

In [10]:
out.groupby('cluster_Names').size().reset_index(name='counts').sort_values(by='counts', ascending=False)

,cluster_Names,counts
1013,DEMS114,547
1018,DEMS124,471
597,"BS I,Bica-Schmitt I",415
1006,DEMS104,322
634,BS153,296
...,...,...
1244,H86-91,1
405,BMS15,1
1246,"H86-98,SOGLE44",1
1247,"H86-99,SOGLE190",1


In [ ]:
# dtypes = {
#     "id": "string",
#     "ra": "float32",
#     "dec": "float32",
#     "gaia_source_id": "string",
#     "parallax": "float32",
#     "gaia_ruwe": "float32",
#     "gaia_parallax_over_error": "float32",
#     "gaia_classprob_dsc_combmod_star": "float32",
#     "gaia_in_qso_candidates": "int8",
#     "gaia_in_galaxy_candidates": "int8",
#     "gaia_phot_bp_rp_excess_factor": "float32",
#     "err_mag_psf_u": "float32",
#     "err_mag_psf_j0378": "float32",
#     "err_mag_psf_j0395": "float32",
#     "err_mag_psf_j0410": "float32",
#     "err_mag_psf_j0430": "float32",
#     "err_mag_psf_g": "float32",
#     "err_mag_psf_j0515": "float32",
#     "err_mag_psf_r": "float32",
#     "err_mag_psf_j0660": "float32",
#     "err_mag_psf_i": "float32",
#     "err_mag_psf_j0861": "float32",
#     "err_mag_psf_z": "float32",
#     # --- magnitudes ---
#     "mag_psf_u": "float32",
#     "mag_psf_j0378": "float32",
#     "mag_psf_j0395": "float32",
#     "mag_psf_j0410": "float32",
#     "mag_psf_j0430": "float32",
#     "mag_psf_g": "float32",
#     "mag_psf_j0515": "float32",
#     "mag_psf_r": "float32",
#     "mag_psf_j0660": "float32",
#     "mag_psf_i": "float32",
#     "mag_psf_j0861": "float32",
#     "mag_psf_z": "float32"
# }


# full = pd.read_csv(
#     "/mnt/hdcasa/splus_gaia/oficial/MC.csv",
#     engine="pyarrow",
#     dtype=dtypes,
#     usecols=list(dtypes.keys())
# )

# full = full[(full['err_mag_psf_j0660'] < 0.2)&
#         (full['err_mag_psf_i'] < 0.2)&
#         (full['err_mag_psf_r'] < 0.2)&
#         (full['mag_psf_j0660']/full['err_mag_psf_j0660'] > 10)&
#         (full['mag_psf_i']/full['err_mag_psf_i'] > 10)&
#         (full['mag_psf_r']/full['err_mag_psf_r'] > 10)&
#         (full['mag_psf_r'].between(13, 19.5))]

# len(full)


# full.to_csv('all_detections_mc.csv', index=False)

In [4]:
full = pd.read_csv('/home/shared/splus_gaia/notebooks/magellanic_clouds/studying_diff_objs/be_stars/all_detections_mc.csv')

In [ ]:
import numpy as np
import pandas as pd

def _radec_to_unitvec(ra_deg, dec_deg):
    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)
    cosd = np.cos(dec)
    x = cosd * np.cos(ra)
    y = cosd * np.sin(ra)
    z = np.sin(dec)
    return np.column_stack([x, y, z]).astype(np.float32)

def _ang_to_chord(theta_rad):
    # chord distance on unit sphere: d = 2 sin(theta/2)
    return 2.0 * np.sin(theta_rad / 2.0)

def match_stars_to_clusters_stream(
    clusters: pd.DataFrame,
    stars: pd.DataFrame,
    *,
    cluster_ra_col="ra",
    cluster_dec_col="dec",
    cluster_amaj_col="amaj",
    stars_ra_col="ra",
    stars_dec_col="dec",
    amaj_unit="arcmin",
    keep_cols_clusters=None,
    keep_cols_stars=None,
    star_chunksize=500_000,
    return_pairs=True,               # if False: only returns aggregated stats per cluster
    out_path=None,                   # if set: appends results to parquet/csv to avoid holding in RAM
):
    """
    Memory-safe spherical crossmatch:
      - builds KDTree on clusters only
      - streams stars in chunks
      - optionally writes output incrementally

    Requires: scipy
    """
    from scipy.spatial import cKDTree

    if keep_cols_clusters is None: keep_cols_clusters = []
    if keep_cols_stars is None: keep_cols_stars = []

    # --- clean clusters ---
    cl_req = [cluster_ra_col, cluster_dec_col, cluster_amaj_col] + keep_cols_clusters
    cl_df = clusters[cl_req].replace([np.inf, -np.inf], np.nan).dropna(
        subset=[cluster_ra_col, cluster_dec_col, cluster_amaj_col]
    )
    cl_df = cl_df[cl_df[cluster_amaj_col] > 0].copy()
    if cl_df.empty:
        return pd.DataFrame()

    # convert cluster radii to radians
    unit_to_deg = {"deg": 1.0, "arcmin": 1/60, "arcsec": 1/3600}
    if amaj_unit not in unit_to_deg:
        raise ValueError(f"amaj_unit must be one of {list(unit_to_deg)}")
    cl_r_deg = cl_df[cluster_amaj_col].to_numpy(dtype=np.float64) * unit_to_deg[amaj_unit]
    cl_r_rad = np.deg2rad(cl_r_deg)

    # build cluster KDTree in 3D
    cl_vec = _radec_to_unitvec(cl_df[cluster_ra_col].to_numpy(), cl_df[cluster_dec_col].to_numpy())
    tree = cKDTree(cl_vec)

    # global search radius (chord distance)
    max_r = float(np.nanmax(cl_r_rad))
    max_chord = _ang_to_chord(max_r)

    # for outputs
    results = []
    # optional aggregation per cluster
    counts = np.zeros(len(cl_df), dtype=np.int64)
    min_sep_arcsec = np.full(len(cl_df), np.inf, dtype=np.float64)

    # stars index array for mapping back to original indices
    st_req = [stars_ra_col, stars_dec_col] + keep_cols_stars
    st_df_all = stars[st_req].replace([np.inf, -np.inf], np.nan).dropna(subset=[stars_ra_col, stars_dec_col])
    if st_df_all.empty:
        return pd.DataFrame()

    st_idx_all = st_df_all.index.to_numpy()

    # stream chunks
    n = len(st_df_all)
    for s0 in range(0, n, star_chunksize):
        s1 = min(n, s0 + star_chunksize)
        chunk = st_df_all.iloc[s0:s1]
        chunk_idx = st_idx_all[s0:s1]

        st_vec = _radec_to_unitvec(chunk[stars_ra_col].to_numpy(), chunk[stars_dec_col].to_numpy())

        # candidate cluster neighbors for each star (indices into cl_df)
        neigh_lists = tree.query_ball_point(st_vec, r=max_chord)

        # flatten (star_i, cl_i) pairs
        if not any(neigh_lists):
            continue

        star_pos = []
        cl_pos = []
        for i, lst in enumerate(neigh_lists):
            if lst:
                star_pos.extend([i] * len(lst))
                cl_pos.extend(lst)

        star_pos = np.asarray(star_pos, dtype=np.int32)
        cl_pos = np.asarray(cl_pos, dtype=np.int32)

        # compute angular separation (exact) via dot product on unit sphere
        dots = np.einsum("ij,ij->i", st_vec[star_pos], cl_vec[cl_pos]).astype(np.float64)
        dots = np.clip(dots, -1.0, 1.0)
        sep_rad = np.arccos(dots)

        # per-cluster radius filter
        keep = sep_rad <= cl_r_rad[cl_pos]
        if not np.any(keep):
            continue

        star_pos = star_pos[keep]
        cl_pos = cl_pos[keep]
        sep_rad = sep_rad[keep]

        # update aggregations
        np.add.at(counts, cl_pos, 1)
        sep_arcsec = np.rad2deg(sep_rad) * 3600.0
        # min sep per cluster
        # (vectorized "min-at" workaround)
        order = np.argsort(cl_pos)
        cl_sorted = cl_pos[order]
        sep_sorted = sep_arcsec[order]
        # scan blocks
        start = 0
        while start < len(cl_sorted):
            cid = cl_sorted[start]
            end = start + 1
            while end < len(cl_sorted) and cl_sorted[end] == cid:
                end += 1
            m = sep_sorted[start:end].min()
            if m < min_sep_arcsec[cid]:
                min_sep_arcsec[cid] = m
            start = end

        if return_pairs:
            out = pd.DataFrame({
                "cluster_index": cl_df.index.to_numpy()[cl_pos],
                "star_index": chunk_idx[star_pos],
                "sep_arcsec": sep_arcsec,
                "sep_arcmin": sep_arcsec / 60.0,
            })

            # attach metadata lazily (only for matched rows)
            cl_out = clusters.loc[out["cluster_index"], [cluster_ra_col, cluster_dec_col, cluster_amaj_col] + keep_cols_clusters]
            cl_out = cl_out.add_prefix("cluster_").reset_index(drop=True)

            st_out = stars.loc[out["star_index"], [stars_ra_col, stars_dec_col] + keep_cols_stars]
            st_out = st_out.add_prefix("star_").reset_index(drop=True)

            out = pd.concat([out.reset_index(drop=True), cl_out, st_out], axis=1)

            if out_path:
                # safest for huge outputs: write per chunk
                if out_path.endswith(".parquet"):
                    out.to_parquet(out_path, index=False, engine="pyarrow", compression="zstd", append=True)
                else:
                    out.to_csv(out_path, index=False, mode="a", header=not np.exists(out_path))
            else:
                results.append(out)

    if return_pairs and not out_path:
        if results:
            return pd.concat(results, ignore_index=True).sort_values("sep_arcmin").reset_index(drop=True)
        return pd.DataFrame()

    # return aggregation per cluster (super small output)
    agg = cl_df[[cluster_ra_col, cluster_dec_col, cluster_amaj_col] + keep_cols_clusters].copy()
    agg["n_stars_matched"] = counts
    agg["min_sep_arcsec"] = np.where(np.isfinite(min_sep_arcsec), min_sep_arcsec, np.nan)
    return agg

In [6]:
# clusters_df: colunas ["name","ra","dec","amaj"] (amaj em arcmin, por ex.)
# emitters_df: colunas ["source_id","ra","dec", ...]
out2 = match_stars_to_clusters_stream(
    bica,
    full,
    cluster_ra_col="ra_deg",
    cluster_dec_col="dec_deg",
    cluster_amaj_col="amaj",
    stars_ra_col="ra",
    stars_dec_col="dec",
    amaj_unit="arcmin",
    keep_cols_clusters=["Names"],         # traga o nome do aglomerado
    keep_cols_stars=["id", "gaia_source_id",  "err_mag_psf_u", "err_mag_psf_j0378", "err_mag_psf_j0395",
       "err_mag_psf_j0410", "err_mag_psf_j0430", "err_mag_psf_g",
       "err_mag_psf_j0515", "err_mag_psf_r", "err_mag_psf_j0660",
       "err_mag_psf_i", "err_mag_psf_j0861", "err_mag_psf_z", "mag_psf_u",
       "mag_psf_j0378", "mag_psf_j0395", "mag_psf_j0410", "mag_psf_j0430",
       "mag_psf_g", "mag_psf_j0515", "mag_psf_r", "mag_psf_j0660", "mag_psf_i",
       "mag_psf_j0861", "mag_psf_z"]  # traga id/atributos da estrela
)

# "out" tem uma linha por estrela-associada, com "cluster_*" e "star_*"

In [7]:
out2

,cluster_index,star_index,sep_arcsec,sep_arcmin,cluster_ra_deg,cluster_dec_deg,cluster_amaj,cluster_Names,star_ra,star_dec,...,star_mag_psf_j0395,star_mag_psf_j0410,star_mag_psf_j0430,star_mag_psf_g,star_mag_psf_j0515,star_mag_psf_r,star_mag_psf_j0660,star_mag_psf_i,star_mag_psf_j0861,star_mag_psf_z
0,1537,9024072,0.000000,0.000000,80.820830,-70.236110,1.0,HS265,80.844990,-70.237880,...,NaN,20.346657,20.113280,19.821210,19.558699,19.120707,18.835772,18.868717,18.609340,18.546880
1,5503,3199723,0.000000,0.000000,20.216667,-72.532778,2.3,B159,20.232445,-72.543860,...,17.553400,17.547718,17.611395,17.765530,17.780540,18.041523,18.180054,18.310547,18.515112,18.548170
2,5503,3199735,0.000000,0.000000,20.216667,-72.532778,2.3,B159,20.198063,-72.540276,...,19.888874,19.545140,19.436460,19.098330,18.894615,18.619143,18.551958,18.428390,18.380102,18.342700
3,5503,3199737,0.000000,0.000000,20.216667,-72.532778,2.3,B159,20.246944,-72.539444,...,19.133230,19.097187,19.151480,19.288197,19.178060,19.423344,19.563780,19.647856,20.026052,19.723524
4,5503,3199750,0.000000,0.000000,20.216667,-72.532778,2.3,B159,20.240545,-72.539055,...,20.762547,20.484926,20.341381,19.798716,19.652264,19.211246,19.153303,18.973180,18.822931,18.822912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1946550,5736,3187722,7199.567853,119.992798,32.045833,-74.446111,120.0,"BS I,Bica-Schmitt I",25.834045,-73.421480,...,20.740316,20.333145,19.954956,19.440840,19.200788,18.879017,18.817780,18.672285,18.555313,18.492603
1946551,5736,4732032,7199.567853,119.992798,32.045833,-74.446111,120.0,"BS I,Bica-Schmitt I",25.834480,-73.421390,...,20.358576,19.825651,19.734732,19.317090,19.168585,18.685099,18.613907,18.395334,18.288310,18.226505
1946552,5736,4723106,7199.920144,119.998669,32.045833,-74.446111,120.0,"BS I,Bica-Schmitt I",25.040113,-73.858470,...,19.687912,19.465380,19.439308,19.209452,19.067453,18.859980,18.852978,18.795944,18.815159,18.771027
1946553,5736,3188064,7199.920144,119.998669,32.045833,-74.446111,120.0,"BS I,Bica-Schmitt I",26.353388,-73.224110,...,20.895376,21.333120,21.069246,20.236404,19.711428,19.280987,19.096510,18.913187,18.633595,18.626700


In [8]:
out2.groupby('cluster_Names').size().reset_index(name='counts').sort_values(by='counts', ascending=False)

,cluster_Names,counts
1162,BS153,71791
2388,DEMS114,56747
1098,"BS I,Bica-Schmitt I",54825
2393,DEMS124,35886
2375,"DEM66,B-OB10",30010
...,...,...
4343,MA106,1
5178,"SL641,BRHT18b,NKN-N160-Y2",1
5173,"SL631,LOGLE655",1
5459,"SOGLE15,BMS100",1
